In [49]:
import duckdb
import os
import json

path = os.path.expanduser("~/projects/statsbomb-data/data/events/3754217.json")

with open(path) as f:
    raw_events = json.load(f)

starting_xi_events = [e for e in raw_events if e['type']['name'] == 'Starting XI']
sub_events = [e for e in raw_events if e['type']['name'] == 'Substitution']

print(f"Starting XI events: {len(starting_xi_events)}")
print(f"Substitution events: {len(sub_events)}")
print()
print(json.dumps(starting_xi_events[0], indent=2, ensure_ascii=False)[:1500])


Starting XI events: 2
Substitution events: 6

{
  "id": "9d86a178-3514-45d1-9d14-1372e846d17b",
  "index": 1,
  "period": 1,
  "timestamp": "00:00:00.000",
  "minute": 0,
  "second": 0,
  "type": {
    "id": 35,
    "name": "Starting XI"
  },
  "possession": 1,
  "possession_team": {
    "id": 33,
    "name": "Chelsea"
  },
  "play_pattern": {
    "id": 1,
    "name": "Regular Play"
  },
  "team": {
    "id": 33,
    "name": "Chelsea"
  },
  "duration": 0.0,
  "tactics": {
    "formation": 4231,
    "lineup": [
      {
        "player": {
          "id": 3339,
          "name": "Asmir Begović"
        },
        "position": {
          "id": 1,
          "name": "Goalkeeper"
        },
        "jersey_number": 1
      },
      {
        "player": {
          "id": 5594,
          "name": "Branislav Ivanović"
        },
        "position": {
          "id": 2,
          "name": "Right Back"
        },
        "jersey_number": 2
      },
      {
        "player": {
          "id": 3456,


In [50]:
##players_id = [e for e in starting_xi_events if e[0]['tactics']['lineup'] == "id"]
##players_name = [e for e in starting_xi_events if e[0]['tactics']['lineup'] == "name"]

##print(players_id)
##print(players_name)

##incorrect

In [51]:
lineup = starting_xi_events[0]['tactics']['lineup']

for player_entry in lineup:
    print(player_entry['player']['id'], player_entry['player']['name'])

3339 Asmir Begović
5594 Branislav Ivanović
3456 Kurt Happy Zouma
3645 Gary Cahill
3957 César Azpilicueta Tanco
3478 Francesc Fàbregas i Soler
3381 Nemanja Matić
3958 Pedro Eliezer Rodríguez Ledesma
40122 Oscar dos Santos Emboaba Júnior
3621 Eden Hazard
5198 Diego da Silva Costa


In [52]:
start_minutes = {}

lineup = starting_xi_events[0]['tactics']['lineup']

for player_entry in lineup:
    start_minutes[player_entry['player']['name']] = 0

print(start_minutes)

{'Asmir Begović': 0, 'Branislav Ivanović': 0, 'Kurt Happy Zouma': 0, 'Gary Cahill': 0, 'César Azpilicueta Tanco': 0, 'Francesc Fàbregas i Soler': 0, 'Nemanja Matić': 0, 'Pedro Eliezer Rodríguez Ledesma': 0, 'Oscar dos Santos Emboaba Júnior': 0, 'Eden Hazard': 0, 'Diego da Silva Costa': 0}


In [53]:
start_minutes = {}
chelsea_lineup = starting_xi_events[0]['tactics']['lineup']
arsenal_lineup = starting_xi_events[1]['tactics']['lineup']

for player_entry in chelsea_lineup:
    start_minutes[player_entry['player']['name']] = 0

for player_entry in arsenal_lineup:
    start_minutes[player_entry['player']['name']] = 0

print(start_minutes)

print()

end_minutes = {}

for sub in sub_events:
    end_minutes[sub['player']['name']] = sub['minute']
    start_minutes[sub['substitution']['replacement']['name']] = sub['minute']

final_minute = max(e['minute'] for e in raw_events)
print(final_minute)

minutes_played = {}
for player in start_minutes:
    player_end = end_minutes.get(player, final_minute) 
    minutes_played[player] = player_end - start_minutes[player]

print(minutes_played)

{'Asmir Begović': 0, 'Branislav Ivanović': 0, 'Kurt Happy Zouma': 0, 'Gary Cahill': 0, 'César Azpilicueta Tanco': 0, 'Francesc Fàbregas i Soler': 0, 'Nemanja Matić': 0, 'Pedro Eliezer Rodríguez Ledesma': 0, 'Oscar dos Santos Emboaba Júnior': 0, 'Eden Hazard': 0, 'Diego da Silva Costa': 0, 'Petr Čech': 0, 'Héctor Bellerín Moruno': 0, 'Gabriel Armando de Abreu': 0, 'Laurent Koscielny': 0, 'Ignacio Monreal Eraso': 0, 'Francis Joseph Coquelin': 0, 'Santiago Cazorla González': 0, 'Aaron Ramsey': 0, 'Mesut Özil': 0, 'Alexis Alejandro Sánchez Sánchez': 0, 'Theo Walcott': 0}

95
{'Asmir Begović': 95, 'Branislav Ivanović': 95, 'Kurt Happy Zouma': 95, 'Gary Cahill': 95, 'César Azpilicueta Tanco': 95, 'Francesc Fàbregas i Soler': 91, 'Nemanja Matić': 95, 'Pedro Eliezer Rodríguez Ledesma': 95, 'Oscar dos Santos Emboaba Júnior': 68, 'Eden Hazard': 95, 'Diego da Silva Costa': 81, 'Petr Čech': 95, 'Héctor Bellerín Moruno': 95, 'Gabriel Armando de Abreu': 95, 'Laurent Koscielny': 95, 'Ignacio Monreal 

In [54]:
import duckdb
import os
import json
import pandas as pd


def get_minutes_played(match_id):
    path = os.path.expanduser(f"~/projects/statsbomb-data/data/events/{match_id}.json")

    with open(path) as f:
        raw_events = json.load(f)

    starting_xi_events = [e for e in raw_events if e['type']['name'] == 'Starting XI']
    sub_events = [e for e in raw_events if e['type']['name'] == 'Substitution']

    start_minutes = {}
    for player_entry in starting_xi_events[0]['tactics']['lineup']:
        start_minutes[player_entry['player']['name']] = 0

    for player_entry in starting_xi_events[1]['tactics']['lineup']:
        start_minutes[player_entry['player']['name']] = 0

    end_minutes = {}

    for sub in sub_events:
        end_minutes[sub['player']['name']] = sub['minute']
        start_minutes[sub['substitution']['replacement']['name']] = sub['minute']

    final_minute = max(e['minute'] for e in raw_events)

    minutes_played = {}
    for player in start_minutes:
        player_end = end_minutes.get(player, final_minute) 
        minutes_played[player] = player_end - start_minutes[player]

    df = pd.DataFrame.from_dict(minutes_played, orient='index')
    df = df.reset_index()
    df.columns = ['player', 'minutes_played']
    df['match_id'] = match_id
    return df


In [55]:
result = get_minutes_played(3754217)
print(result)

                              player  minutes_played  match_id
0                      Asmir Begović              95   3754217
1                 Branislav Ivanović              95   3754217
2                   Kurt Happy Zouma              95   3754217
3                        Gary Cahill              95   3754217
4            César Azpilicueta Tanco              95   3754217
5          Francesc Fàbregas i Soler              91   3754217
6                      Nemanja Matić              95   3754217
7    Pedro Eliezer Rodríguez Ledesma              95   3754217
8    Oscar dos Santos Emboaba Júnior              68   3754217
9                        Eden Hazard              95   3754217
10              Diego da Silva Costa              81   3754217
11                         Petr Čech              95   3754217
12            Héctor Bellerín Moruno              95   3754217
13          Gabriel Armando de Abreu              95   3754217
14                 Laurent Koscielny              95   

In [ ]:
output_path = os.path.expanduser("~/projects/rolefit/data/processed/match_3754217_minutes.parquet")
os.makedirs(os.path.dirname(output_path), exist_ok=True)
result.to_parquet(output_path)